# EN3150 Assignment 03 — Two-Digit MNIST Classification

## Introduction / Assignment Overview

This notebook implements the complete two-digit MNIST classification workflow. The task is formulated as a **100-class classification problem**, where the two individual digits are combined to form one target label from `00` to `99`.

The notebook is organized in the order of the assignment workflow: project setup, dataset preparation, two CNN models, optimizer comparison, computational-cost analysis, transfer learning, final comparison, error analysis, and results export.

**Notebook convention:** each subsection first explains the purpose and important choices, followed by the relevant implementation and result cells.

# 1. Project Setup

### 1.1 Import Libraries

These cells import the libraries used for data processing, image handling, visualization, PyTorch training, evaluation metrics, and transfer learning.

In [ ]:
# Optional: install missing packages in a Jupyter/VS Code notebook.
# Run only if an import fails.
# %pip install torch torchvision pandas numpy matplotlib scikit-learn pillow tqdm

In [ ]:
import os
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights,
    efficientnet_b0,
    EfficientNet_B0_Weights,
)

warnings.filterwarnings("ignore")

### 1.2 Reproducibility

A fixed random seed is applied to Python, NumPy, and PyTorch so repeated experiments are as consistent as possible.

In [ ]:
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

print("Random seed:", SEED)

### 1.3 Device Configuration

The notebook selects CUDA when an available GPU is detected; otherwise it uses the CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 2. Dataset Overview and Data Loading

### 2.1 Configuration

This cell defines the dataset path, image size, number of classes, batch size, training parameters, and output settings.

In [ ]:
# ============================================================
# CHANGE THIS PATH IF NECESSARY
# ============================================================

DATA_DIR = Path(r"C:\Users\dilha\Downloads\archive")

# Windows example:
# DATA_DIR = Path(r"C:\Users\Sandeepa\Desktop\EN3150_Project\mnist-2-digit-dataset")

IMAGE_SIZE = 64
NUM_CLASSES = 100

BATCH_SIZE = 128 if torch.cuda.is_available() else 32
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

MAIN_EPOCHS = 20
OPTIMIZER_COMPARE_EPOCHS = 5

LEARNING_RATE = 1e-3
MOMENTUM = 0.9

# Set these to False if you want to skip the long SOTA stage initially.
RUN_MOBILENET = True
RUN_EFFICIENTNET = True
SOTA_EPOCHS = 20

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("IMAGE_SIZE:", IMAGE_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)

### 2.2 Verify Dataset Structure

The configured directory is checked before any CSV or image data are loaded.

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR does not exist: {DATA_DIR.resolve()}\n"
        "Please download/extract the Kaggle MNIST 2 Digit Dataset and update DATA_DIR."
    )

print("Dataset contents:")
for item in sorted(DATA_DIR.iterdir()):
    print(" -", item.name)

print("\nCSV files found:")
for p in DATA_DIR.rglob("*.csv"):
    print(" -", p)

### 2.3 Load CSV Files

The training and test CSV files are located and loaded into pandas DataFrames.

In [ ]:
def find_csv(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} inside {DATA_DIR.resolve()}"
        )
    return matches[0]

train_csv = find_csv("train.csv")
test_csv = find_csv("test.csv")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

print("train.csv:", train_csv)
print("test.csv :", test_csv)

print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Test columns :", test_df.columns.tolist())

display(train_df.head())

### 2.4 Check Required Columns

The CSV files are checked for the filename and two digit-label columns required by the assignment.

In [ ]:
required_columns = {"file_name", "label_1", "label_2"}

for name, df in [("train.csv", train_df), ("test.csv", test_df)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(
            f"{name} is missing columns: {missing}. "
            f"Found columns: {df.columns.tolist()}"
        )

print("Required columns are present.")

### 2.5 Create 100-Class Labels

The two individual digits are combined into one class label from 00 to 99.

In [ ]:
def create_two_digit_label(df):
    df = df.copy()

    df["label_1"] = pd.to_numeric(df["label_1"], errors="raise").astype(int)
    df["label_2"] = pd.to_numeric(df["label_2"], errors="raise").astype(int)

    if not df["label_1"].between(0, 9).all():
        raise ValueError("label_1 contains values outside 0-9.")
    if not df["label_2"].between(0, 9).all():
        raise ValueError("label_2 contains values outside 0-9.")

    df["label"] = df["label_1"] * 10 + df["label_2"]
    return df

train_df = create_two_digit_label(train_df)
test_df = create_two_digit_label(test_df)

print("Example labels:")
display(train_df[["file_name", "label_1", "label_2", "label"]].head(15))

### 2.6 Index Image Files

All image files are indexed and matched to the CSV filenames so the Dataset can load the correct image.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

image_lookup = {}

for path in DATA_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        image_lookup[path.name] = str(path)

print("Images indexed:", len(image_lookup))

print("\nFirst 10 image paths:")
for name, path in list(image_lookup.items())[:10]:
    print(name, "->", path)

def add_image_path(df):
    df = df.copy()
    df["image_path"] = df["file_name"].map(image_lookup)
    missing = df["image_path"].isna().sum()

    if missing:
        examples = df.loc[df["image_path"].isna(), "file_name"].head(10).tolist()
        raise FileNotFoundError(
            f"{missing} image files from the CSV were not found. "
            f"Examples: {examples}"
        )
    return df

train_df = add_image_path(train_df)
test_df = add_image_path(test_df)

print("\nAll CSV image filenames were matched successfully.")

# 3. Dataset Exploration and Preprocessing

### 3.1 Dataset Statistics

A basic summary of the available labelled images is generated before splitting the data.

In [ ]:
print("Training CSV samples:", len(train_df))
print("Test CSV samples:", len(test_df))
print("Combined unique images:", pd.concat([train_df, test_df])["file_name"].nunique())
print("Number of classes:", pd.concat([train_df, test_df])["label"].nunique())

### 3.2 Visualize Sample Images

A sample of the actual two-digit images is displayed with its combined target label.

In [ ]:
def show_samples(df, n=15, seed=SEED):
    sample_df = df.sample(n=min(n, len(df)), random_state=seed)

    rows = int(np.ceil(len(sample_df) / 5))
    fig, axes = plt.subplots(rows, 5, figsize=(12, 2.8 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        image = Image.open(row["image_path"]).convert("L")
        ax.imshow(image, cmap="gray")
        ax.set_title(f"Target: {row['label']:02d}")
        ax.axis("off")

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples(train_df, n=15)

### 3.3 Build 70/15/15 Train/Validation/Test Split

The complete labelled image set is split into 70% training, 15% validation, and 15% test data using stratification.

In [ ]:
all_df = pd.concat([train_df, test_df], ignore_index=True)

all_df = all_df.drop_duplicates(subset=["file_name"]).reset_index(drop=True)

print("Total labelled images:", len(all_df))
print("Unique filenames:", all_df["file_name"].nunique())

train_split, temp_split = train_test_split(
    all_df,
    test_size=0.30,
    random_state=SEED,
    stratify=all_df["label"],
)

val_split, test_split = train_test_split(
    temp_split,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_split["label"],
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_split = test_split.reset_index(drop=True)

print("Train:", len(train_split), f"({len(train_split)/len(all_df)*100:.2f}%)")
print("Val  :", len(val_split), f"({len(val_split)/len(all_df)*100:.2f}%)")
print("Test :", len(test_split), f"({len(test_split)/len(all_df)*100:.2f}%)")

### 3.4 Check Class Distribution

The distribution across the three subsets is inspected to verify the split.

In [ ]:
class_counts = pd.DataFrame({
    "train": train_split["label"].value_counts().sort_index(),
    "validation": val_split["label"].value_counts().sort_index(),
    "test": test_split["label"].value_counts().sort_index(),
}).fillna(0).astype(int)

display(class_counts.head(20))

print("Number of classes represented:", all_df["label"].nunique())
print("Expected number of classes:", NUM_CLASSES)

### 3.5 Image Preprocessing

Images are resized, converted to tensors, normalized, and mildly augmented for training.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAffine(
        degrees=8,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

### 3.6 PyTorch Dataset

The custom Dataset class reads an image path and returns a transformed image tensor with its two-digit target.

In [ ]:
class TwoDigitDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("L")

        if self.transform:
            image = self.transform(image)

        label = int(row["label"])
        return image, label

### 3.7 DataLoaders

The dataset objects are converted into training, validation, and test DataLoaders.

In [ ]:
train_dataset = TwoDigitDataset(train_split, train_transform)
val_dataset = TwoDigitDataset(val_split, transform)
test_dataset = TwoDigitDataset(test_split, transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

### 3.8 Check One Batch

A batch is inspected to confirm tensor dimensions, labels, and image values before training.

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", [f"{x:02d}" for x in labels[:10].tolist()])

plt.figure(figsize=(4, 4))
plt.imshow(images[0].squeeze().numpy(), cmap="gray")
plt.title(f"Target: {labels[0].item():02d}")
plt.axis("off")
plt.show()

# 4. Model A — Standard CNN

### 4.1 Model Architecture

Model A is the standard CNN baseline using conventional convolutional layers.

In [ ]:
class StandardCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
model_a = StandardCNN().to(device)
print(model_a)

### 4.2 Parameter Count

The trainable parameter count of the baseline model is measured before training.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_all_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
params_a = count_parameters(model_a)
print(f"Model A trainable parameters: {params_a:,}")

### 4.3 Training Configuration

The common training utilities are separated into small cells so that each function has a clear role.

In [ ]:
criterion = nn.CrossEntropyLoss()
print("Loss function: CrossEntropyLoss")

In [ ]:
def create_optimizer(model, name, lr=LEARNING_RATE, momentum=MOMENTUM):
    name = name.lower()

    if name == "adam":
        return optim.Adam(model.parameters(), lr=lr)

    if name == "sgd":
        return optim.SGD(model.parameters(), lr=lr)

    if name in ["sgd_momentum", "momentum"]:
        return optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum
        )

    raise ValueError(f"Unknown optimizer: {name}")

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            if is_training:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_predictions)

    return epoch_loss, epoch_accuracy

In [ ]:
def train_model(model, train_loader, val_loader, epochs,
                optimizer_name="adam", lr=LEARNING_RATE,
                momentum=MOMENTUM, model_name="model"):

    model = model.to(device)

    optimizer = create_optimizer(
        model, optimizer_name, lr=lr, momentum=momentum
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
        "epoch_time": [],
    }

    best_state = copy.deepcopy(model.state_dict())
    best_val_accuracy = -1.0

    total_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        epoch_start = time.perf_counter()

        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc = run_epoch(
            model, val_loader, criterion
        )

        epoch_time = time.perf_counter() - epoch_start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)
        history["epoch_time"].append(epoch_time)

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"{model_name} | Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc*100:.2f}% | "
            f"Time: {epoch_time:.1f}s"
        )

    total_time = time.perf_counter() - total_start

    model.load_state_dict(best_state)

    history["total_time"] = total_time
    history["avg_epoch_time"] = total_time / epochs
    history["best_val_accuracy"] = best_val_accuracy

    return model, history

In [ ]:
def predict_model(model, loader):
    model.eval()

    all_predictions = []
    all_targets = []

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)

        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(targets.numpy())

    return np.array(all_targets), np.array(all_predictions)

In [ ]:
def evaluate_model(model, loader, model_name="Model"):
    y_true, y_pred = predict_model(model, loader)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )

    cm = confusion_matrix(
        y_true, y_pred, labels=np.arange(NUM_CLASSES)
    )

    print(f"\n{model_name} Test Results")
    print("=" * 50)
    print(f"Accuracy : {accuracy*100:.2f}%")
    print(f"Precision: {precision*100:.2f}% (macro)")
    print(f"Recall   : {recall*100:.2f}% (macro)")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "confusion_matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred,
    }

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Training loss")
    plt.plot(epochs, history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs,
        np.array(history["train_accuracy"]) * 100,
        label="Training accuracy"
    )
    plt.plot(
        epochs,
        np.array(history["val_accuracy"]) * 100,
        label="Validation accuracy"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def save_model_size(model, filename):
    path = OUTPUT_DIR / filename
    torch.save(model.state_dict(), path)
    size_kb = path.stat().st_size / 1024
    size_mb = size_kb / 1024
    return path, size_kb, size_mb

### 4.4 Train Model A

Model A is trained using Adam and the main training configuration.

In [ ]:
class StandardCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model_a = StandardCNN().to(device)
print(model_a)

### 4.5 Evaluate Model A

The trained baseline is evaluated on the held-out test set.

In [ ]:
results_a = evaluate_model(
    model_a,
    test_loader,
    model_name="Model A — Standard CNN"
)

### 4.6 Confusion Matrix

The baseline confusion matrix is displayed to inspect classification errors across the 100 classes.

In [ ]:
def plot_confusion_matrix(cm, title, max_labels=None):
    plt.figure(figsize=(10, 9))
    plt.imshow(cm, interpolation="nearest", aspect="auto")
    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.colorbar()

    if max_labels is not None and max_labels <= cm.shape[0]:
        ticks = np.linspace(0, cm.shape[0]-1, max_labels, dtype=int)
        plt.xticks(ticks)
        plt.yticks(ticks)

    plt.tight_layout()
    plt.show()

plot_confusion_matrix(
    results_a["confusion_matrix"],
    "Model A — 100-Class Confusion Matrix",
    max_labels=20
)

# 5. Model B — Resource-Constrained CNN

### 5.1 Motivation

Model B is designed to achieve a much smaller trainable parameter count while still performing the same 100-class task.

### 5.2 Depthwise-Separable Convolution

The depthwise and pointwise operations are implemented as a reusable building block.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2

        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            padding=padding,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            bias=False,
        )

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

### 5.3 Model Architecture

The lightweight CNN is assembled from depthwise-separable convolution blocks.

In [ ]:
class LightweightCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            DepthwiseSeparableConv(1, 16),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(16, 32),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [ ]:
model_b = LightweightCNN().to(device)
print(model_b)

### 5.4 Parameter Count Verification (<100,000)

The lightweight model is explicitly checked against the assignment's parameter constraint.

In [ ]:
params_b = count_parameters(model_b)

print(f"Model A trainable parameters: {count_parameters(model_a):,}")
print(f"Model B trainable parameters: {params_b:,}")

assert params_b < 100_000, (
    f"Model B has {params_b:,} parameters, which violates the <100,000 requirement."
)

print("\nPASS: Model B is below 100,000 trainable parameters.")

### 5.5 Train Model B

Model B is trained using the same main training procedure as Model A.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2

        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            padding=padding,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            bias=False,
        )

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class LightweightCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            DepthwiseSeparableConv(1, 16),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(16, 32),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model_b = LightweightCNN().to(device)
print(model_b)

### 5.6 Evaluate Model B

The trained lightweight model is evaluated using the same held-out test set.

In [ ]:
results_b = evaluate_model(
    model_b,
    test_loader,
    model_name="Model B — Lightweight CNN"
)

### 5.7 Confusion Matrix

The Model B confusion matrix is displayed for comparison with Model A.

In [ ]:
plot_confusion_matrix(
    results_b["confusion_matrix"],
    "Model B — 100-Class Confusion Matrix",
    max_labels=20
)

# 6. Optimizer Comparison

### 6.1 Adam

Adam is tested first using the lightweight CNN as the comparison architecture.

In [ ]:
optimizer_results = []

In [ ]:
print("\n" + "=" * 70)
print("Optimizer:", "adam")

seed_everything()

temp_model = LightweightCNN().to(device)

optimizer_name = "adam"
lr = 0.001

temp_model, temp_history = train_model(
    temp_model,
    train_loader,
    val_loader,
    epochs=OPTIMIZER_COMPARE_EPOCHS,
    optimizer_name=optimizer_name,
    lr=lr,
    momentum=MOMENTUM,
    model_name=f"Model B - {optimizer_name}",
)

optimizer_results.append({
    "Optimizer": optimizer_name,
    "Learning Rate": lr,
    "Momentum": MOMENTUM if optimizer_name == "sgd_momentum" else 0.0,
    "Best Validation Accuracy (%)":
        temp_history["best_val_accuracy"] * 100,
    "Average Epoch Time (s)":
        temp_history["avg_epoch_time"],
})

### 6.2 SGD

Plain SGD is tested with the selected SGD learning rate.

In [ ]:
print("\n" + "=" * 70)
print("Optimizer:", "sgd")

seed_everything()

temp_model = LightweightCNN().to(device)

optimizer_name = "sgd"
lr = 0.01

temp_model, temp_history = train_model(
    temp_model,
    train_loader,
    val_loader,
    epochs=OPTIMIZER_COMPARE_EPOCHS,
    optimizer_name=optimizer_name,
    lr=lr,
    momentum=MOMENTUM,
    model_name=f"Model B - {optimizer_name}",
)

optimizer_results.append({
    "Optimizer": optimizer_name,
    "Learning Rate": lr,
    "Momentum": MOMENTUM if optimizer_name == "sgd_momentum" else 0.0,
    "Best Validation Accuracy (%)":
        temp_history["best_val_accuracy"] * 100,
    "Average Epoch Time (s)":
        temp_history["avg_epoch_time"],
})

### 6.3 SGD + Momentum

SGD with momentum is tested using the same base learning rate and the configured momentum value.

In [ ]:
print("\n" + "=" * 70)
print("Optimizer:", "sgd_momentum")

seed_everything()

temp_model = LightweightCNN().to(device)

optimizer_name = "sgd_momentum"
lr = 0.01

temp_model, temp_history = train_model(
    temp_model,
    train_loader,
    val_loader,
    epochs=OPTIMIZER_COMPARE_EPOCHS,
    optimizer_name=optimizer_name,
    lr=lr,
    momentum=MOMENTUM,
    model_name=f"Model B - {optimizer_name}",
)

optimizer_results.append({
    "Optimizer": optimizer_name,
    "Learning Rate": lr,
    "Momentum": MOMENTUM if optimizer_name == "sgd_momentum" else 0.0,
    "Best Validation Accuracy (%)":
        temp_history["best_val_accuracy"] * 100,
    "Average Epoch Time (s)":
        temp_history["avg_epoch_time"],
})

### 6.4 Comparison Results

The three optimizer results are combined into one table.

In [ ]:
optimizer_results_df = pd.DataFrame(optimizer_results)
display(optimizer_results_df)

### 6.5 Interpretation

Compare the validation accuracy and average epoch time across the three optimizers. The table provides the numerical evidence for the report discussion.

# 7. Model Size and Computational Cost

### 7.1 Model Size

The trained state dictionaries are saved and their file sizes are measured.

In [ ]:
path_a, size_a_kb, size_a_mb = save_model_size(
    model_a, "model_a_standard_cnn.pth"
)

path_b, size_b_kb, size_b_mb = save_model_size(
    model_b, "model_b_lightweight_cnn.pth"
)

print(f"Model A size: {size_a_kb:.2f} KB ({size_a_mb:.4f} MB)")
print(f"Model B size: {size_b_kb:.2f} KB ({size_b_mb:.4f} MB)")

### 7.2 Training Time

Average training time per epoch recorded during training is compared for Model A and Model B.

In [ ]:
display(
    custom_results[
        ["Model", "Training Time / Epoch (s)"]
    ].style.format({
        "Training Time / Epoch (s)": "{:.2f}"
    })
)

### 7.3 Parameter Count Comparison

A compact table combines the parameter count, storage size, training time, and test performance of the two custom CNNs.

In [ ]:
custom_results = pd.DataFrame([
    {
        "Model": "Model A - Standard CNN",
        "Trainable Parameters": count_parameters(model_a),
        "Model Size (KB)": size_a_kb,
        "Model Size (MB)": size_a_mb,
        "Training Time / Epoch (s)": history_a["avg_epoch_time"],
        "Test Accuracy (%)": results_a["accuracy"] * 100,
        "Precision (%)": results_a["precision"] * 100,
        "Recall (%)": results_a["recall"] * 100,
    },
    {
        "Model": "Model B - Lightweight CNN",
        "Trainable Parameters": count_parameters(model_b),
        "Model Size (KB)": size_b_kb,
        "Model Size (MB)": size_b_mb,
        "Training Time / Epoch (s)": history_b["avg_epoch_time"],
        "Test Accuracy (%)": results_b["accuracy"] * 100,
        "Precision (%)": results_b["precision"] * 100,
        "Recall (%)": results_b["recall"] * 100,
    }
])

display(custom_results)

# 8. Transfer Learning

### 8.1 MobileNetV2

A separate three-channel preprocessing pipeline is created because the pretrained torchvision models expect RGB-style inputs.

In [ ]:
USE_PRETRAINED = True

sota_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

sota_train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAffine(
        degrees=8,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

sota_train_dataset = TwoDigitDataset(train_split, sota_train_transform)
sota_val_dataset = TwoDigitDataset(val_split, sota_transform)
sota_test_dataset = TwoDigitDataset(test_split, sota_transform)

sota_train_loader = DataLoader(
    sota_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sota_val_loader = DataLoader(
    sota_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sota_test_loader = DataLoader(
    sota_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("SOTA dataloaders ready.")

### 8.2 Fine-tuning MobileNetV2

The MobileNetV2 architecture is adapted to the 100-class output and fine-tuned on the prepared dataset.

In [ ]:
def build_mobilenet(num_classes=NUM_CLASSES, pretrained=True):
    if pretrained:
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    else:
        model = mobilenet_v2(weights=None)

    model.classifier[1] = nn.Linear(
        model.last_channel,
        num_classes
    )

    return model

In [ ]:
mobilenet_model = build_mobilenet(
    pretrained=USE_PRETRAINED
).to(device)

print("MobileNetV2 parameters:", f"{count_all_parameters(mobilenet_model):,}")

### 8.3 Evaluation

The trained MobileNetV2 model is evaluated and its saved state size is recorded.

In [ ]:
mobilenet_results = None
mobilenet_size_kb = None
mobilenet_size_mb = None

if RUN_MOBILENET:
    mobilenet_results = evaluate_model(
        mobilenet_model,
        sota_test_loader,
        model_name="MobileNetV2"
    )

    _, mobilenet_size_kb, mobilenet_size_mb = save_model_size(
        mobilenet_model,
        "mobilenet_v2_two_digit.pth"
    )

### 8.4 EfficientNet-B0

EfficientNet-B0 uses the same prepared three-channel data pipeline and the same 100-class target.

In [ ]:
def build_efficientnet(num_classes=NUM_CLASSES, pretrained=True):
    if pretrained:
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    else:
        model = efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(
        in_features,
        num_classes
    )

    return model

In [ ]:
efficientnet_model = build_efficientnet(
    pretrained=USE_PRETRAINED
).to(device)

print("EfficientNet-B0 parameters:", f"{count_all_parameters(efficientnet_model):,}")

### 8.5 Fine-tuning EfficientNet-B0

The EfficientNet-B0 model is trained using the shared training utility.

In [ ]:
efficientnet_history = None

if RUN_EFFICIENTNET:
    seed_everything()

    efficientnet_model, efficientnet_history = train_model(
        efficientnet_model,
        sota_train_loader,
        sota_val_loader,
        epochs=SOTA_EPOCHS,
        optimizer_name="adam",
        lr=1e-4 if USE_PRETRAINED else 1e-3,
        model_name="EfficientNet-B0",
    )

    plot_history(efficientnet_history, "EfficientNet-B0")
else:
    print("EfficientNet-B0 stage skipped. Set RUN_EFFICIENTNET=True to run it.")

### 8.6 Evaluation

The EfficientNet-B0 model is evaluated and its state size is recorded for the final comparison.

In [ ]:
efficientnet_results = None
efficientnet_size_kb = None
efficientnet_size_mb = None

if RUN_EFFICIENTNET:
    efficientnet_results = evaluate_model(
        efficientnet_model,
        sota_test_loader,
        model_name="EfficientNet-B0"
    )

    _, efficientnet_size_kb, efficientnet_size_mb = save_model_size(
        efficientnet_model,
        "efficientnet_b0_two_digit.pth"
    )

# 9. Final Model Comparison

### 9.1 Accuracy Comparison

The final comparison table is created from all available model results, including optional transfer-learning models when their runs are enabled.

In [ ]:
comparison_rows = []

comparison_rows.append({
    "Model": "Model A - Standard CNN",
    "Parameters": count_parameters(model_a),
    "Size (KB)": size_a_kb,
    "Size (MB)": size_a_mb,
    "Time/Epoch (s)": history_a["avg_epoch_time"],
    "Accuracy (%)": results_a["accuracy"] * 100,
    "Precision (%)": results_a["precision"] * 100,
    "Recall (%)": results_a["recall"] * 100,
})

comparison_rows.append({
    "Model": "Model B - Lightweight CNN",
    "Parameters": count_parameters(model_b),
    "Size (KB)": size_b_kb,
    "Size (MB)": size_b_mb,
    "Time/Epoch (s)": history_b["avg_epoch_time"],
    "Accuracy (%)": results_b["accuracy"] * 100,
    "Precision (%)": results_b["precision"] * 100,
    "Recall (%)": results_b["recall"] * 100,
})

if mobilenet_results is not None:
    comparison_rows.append({
        "Model": "MobileNetV2",
        "Parameters": count_all_parameters(mobilenet_model),
        "Size (KB)": mobilenet_size_kb,
        "Size (MB)": mobilenet_size_mb,
        "Time/Epoch (s)": mobilenet_history["avg_epoch_time"],
        "Accuracy (%)": mobilenet_results["accuracy"] * 100,
        "Precision (%)": mobilenet_results["precision"] * 100,
        "Recall (%)": mobilenet_results["recall"] * 100,
    })

if efficientnet_results is not None:
    comparison_rows.append({
        "Model": "EfficientNet-B0",
        "Parameters": count_all_parameters(efficientnet_model),
        "Size (KB)": efficientnet_size_kb,
        "Size (MB)": efficientnet_size_mb,
        "Time/Epoch (s)": efficientnet_history["avg_epoch_time"],
        "Accuracy (%)": efficientnet_results["accuracy"] * 100,
        "Precision (%)": efficientnet_results["precision"] * 100,
        "Recall (%)": efficientnet_results["recall"] * 100,
    })

final_comparison = pd.DataFrame(comparison_rows)

display(
    final_comparison.style.format({
        "Parameters": "{:,.0f}",
        "Size (KB)": "{:.2f}",
        "Size (MB)": "{:.4f}",
        "Time/Epoch (s)": "{:.2f}",
        "Accuracy (%)": "{:.2f}",
        "Precision (%)": "{:.2f}",
        "Recall (%)": "{:.2f}",
    })
)

final_comparison.to_csv(
    OUTPUT_DIR / "final_model_comparison.csv",
    index=False
)

In [ ]:
display(
    final_comparison[
        ["Model", "Accuracy (%)"]
    ].style.format({"Accuracy (%)": "{:.2f}"})
)

### 9.2 Precision and Recall

Macro precision and recall are compared across the available models.

In [ ]:
display(
    final_comparison[
        ["Model", "Precision (%)", "Recall (%)"]
    ].style.format({
        "Precision (%)": "{:.2f}",
        "Recall (%)": "{:.2f}"
    })
)

### 9.3 Model Size

Parameter counts and saved model sizes are compared.

In [ ]:
display(
    final_comparison[
        ["Model", "Parameters", "Size (KB)", "Size (MB)"]
    ].style.format({
        "Parameters": "{:,.0f}",
        "Size (KB)": "{:.2f}",
        "Size (MB)": "{:.4f}"
    })
)

### 9.4 Computational Cost

Training time per epoch is compared as a practical computational-cost measure.

In [ ]:
display(
    final_comparison[
        ["Model", "Time/Epoch (s)"]
    ].style.format({"Time/Epoch (s)": "{:.2f}"})
)

### 9.5 Accuracy vs Model Size

The scatter plot shows the trade-off between predictive accuracy and model storage size.

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    final_comparison["Size (MB)"],
    final_comparison["Accuracy (%)"],
    s=100
)

for _, row in final_comparison.iterrows():
    plt.annotate(
        row["Model"],
        (row["Size (MB)"], row["Accuracy (%)"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Model size (MB)")
plt.ylabel("Test accuracy (%)")
plt.title("Accuracy vs Model Size")
plt.grid(True)
plt.tight_layout()
plt.show()

### Report Summary

This summary identifies the highest-accuracy and smallest models and reports the training-time measurements used in the discussion.

In [ ]:
best_accuracy_row = final_comparison.loc[
    final_comparison["Accuracy (%)"].idxmax()
]

smallest_model_row = final_comparison.loc[
    final_comparison["Parameters"].idxmin()
]

print("REPORT SUMMARY")
print("=" * 70)

print(
    f"Highest test accuracy: {best_accuracy_row['Model']} "
    f"({best_accuracy_row['Accuracy (%)']:.2f}%)"
)

print(
    f"Smallest model by parameter count: "
    f"{smallest_model_row['Model']} "
    f"({smallest_model_row['Parameters']:,} parameters)"
)

print(
    f"Model B parameters: {count_parameters(model_b):,} "
    f"(must be <100,000)"
)

print(
    f"Model A average training time/epoch: "
    f"{history_a['avg_epoch_time']:.2f} s"
)

print(
    f"Model B average training time/epoch: "
    f"{history_b['avg_epoch_time']:.2f} s"
)

# 10. Error Analysis

### 10.1 Most Common Classification Confusions

The largest off-diagonal confusion-matrix entries are extracted for Model A and Model B.

In [ ]:
def top_confusions(cm, top_n=20):
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)

    flat_indices = np.argsort(cm_copy.ravel())[::-1]

    rows = []
    used = 0

    for flat_idx in flat_indices:
        true_class, predicted_class = np.unravel_index(
            flat_idx, cm_copy.shape
        )

        count = cm_copy[true_class, predicted_class]

        if count <= 0:
            break

        rows.append({
            "True": f"{true_class:02d}",
            "Predicted": f"{predicted_class:02d}",
            "Count": int(count),
        })

        used += 1

        if used >= top_n:
            break

    return pd.DataFrame(rows)

print("Model A top confusions:")
display(top_confusions(results_a["confusion_matrix"]))

print("Model B top confusions:")
display(top_confusions(results_b["confusion_matrix"]))

### 10.2 Interpretation

The most common confusion pairs can be discussed as examples of visually difficult or similar two-digit classes.

# 11. Results Export

### 11.1 Save Figures

The figures are generated in their corresponding analysis sections; the output directory is confirmed here.

In [ ]:
print("Results output directory:")
print(OUTPUT_DIR.resolve())

### 11.2 Save CSV Results

The main experiment tables are exported as CSV files for use in the report.

In [ ]:
custom_results.to_csv(
    OUTPUT_DIR / "custom_model_results.csv",
    index=False
)

final_comparison.to_csv(
    OUTPUT_DIR / "final_model_comparison.csv",
    index=False
)

optimizer_results_df.to_csv(
    OUTPUT_DIR / "optimizer_comparison.csv",
    index=False
)

print("CSV result files saved.")

### 11.3 Save Model Weights

The available trained model-weight files are checked and listed.

In [ ]:
print("Model-weight files:")
for filename in [
    "model_a_standard_cnn.pth",
    "model_b_lightweight_cnn.pth",
    "mobilenet_v2_two_digit.pth",
    "efficientnet_b0_two_digit.pth",
]:
    file_path = OUTPUT_DIR / filename
    print(f" - {filename}: {'FOUND' if file_path.exists() else 'not generated'}")

### Final Output Directory Contents

A final directory listing shows all generated artifacts.

In [ ]:
print("Output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name:45s} {path.stat().st_size/1024:.2f} KB")

# 12. Assignment Checklist

Use this final checklist to confirm that the notebook contains the major assignment components before submission.

**Dataset**
- 100-class two-digit label construction
- Dataset validation and image indexing
- 70/15/15 stratified train/validation/test split
- Image preprocessing and PyTorch DataLoaders

**Models**
- Standard CNN (Model A)
- Resource-constrained CNN (Model B)
- Verification of Model B's `< 100,000` trainable parameters

**Experiments**
- Model training and evaluation
- Optimizer comparison: Adam, SGD, SGD + Momentum
- Model size and training-time comparison
- MobileNetV2 and EfficientNet-B0 transfer learning

**Analysis**
- Accuracy, precision, recall
- Confusion matrices
- Model-size/accuracy trade-off
- Common classification confusions
- Exported model weights and result artifacts

Run the notebook from top to bottom before final submission so that all cells execute in the intended dependency order.